# DriveWise: Metadata-Aware-Automotive-RAG-Assistant

This notebook shows how the DriveWise pipeline works on car brochure text. It only uses the brochure for the selected car, so the answer stays specific to that model.

What it does:
- split brochures into sections
- create embeddings and store them in FAISS
- filter by brand and model before search
- re-rank the search results
- generate an answer from the best match
- log and evaluate the result

The main code is in `src/`, and this notebook just runs it.

## Step 0: Install dependencies

If needed, uncomment the cell below and run it once. The embedding, re-ranking, and generation models are downloaded from HuggingFace the first time they are used.

In [1]:
# !pip install sentence-transformers faiss-cpu transformers torch pandas streamlit PyMuPDF


## Step 1: Load and chunk the brochures

`src/chunker.py` reads each `.txt` file in `data/brochures/` and splits it at `[SECTION: ...]` markers. Each chunk stays on one topic, so the model sees complete information instead of broken fragments.

Each chunk also stores metadata such as brand, model, section, page number, and brochure version. That metadata lets us narrow the search to one car before doing similarity matching.

In [2]:
import pandas as pd
from src.chunker import load_all_brochures

chunks = load_all_brochures("data/brochures")
df = pd.DataFrame(chunks)
print(f"Loaded {len(df)} chunks from {df['source_file'].nunique()} brochures")
df.head()


[chunker] Loaded PDF: hyundai_creta.pdf  (16 chunks)
[chunker] Loaded PDF: maruti_baleno.pdf  (8 chunks)
[chunker] Loaded PDF: tata_nexon.pdf  (34 chunks)
Loaded 58 chunks from 3 brochures


,text,brand,model,section,version,source_file,page_number,chunk_id
0,Undisputed. Ultimate. \nNothing stands in comp...,Hyundai,Creta,exterior and design,official,hyundai_creta.pdf,2,hyundai_creta.pdf::exterior and design::p2
1,Other features: Puddle lamp with welcome funct...,Hyundai,Creta,exterior and design,official,hyundai_creta.pdf,4,hyundai_creta.pdf::exterior and design::p4
2,Discover spatial artistry. \nOverwhelming pr...,Hyundai,Creta,interior and comfort,official,hyundai_creta.pdf,5,hyundai_creta.pdf::interior and comfort::p5
3,Always forward thinking.\nStand above it all a...,Hyundai,Creta,infotainment and connectivity,official,hyundai_creta.pdf,6,hyundai_creta.pdf::infotainment and connectivi...
4,Forward Collision - Avoidance Assist\nJunction...,Hyundai,Creta,safety,official,hyundai_creta.pdf,7,hyundai_creta.pdf::safety::p7


## Step 2: Embedding model + FAISS vector database

`src/retriever.py` uses `all-MiniLM-L6-v2` to turn each chunk into a 384-dimensional vector. Similar chunks stay close together even when the wording is different.

The vectors are stored in a FAISS `IndexFlatIP` index for fast similarity search.

In [3]:
from src.retriever import VectorStore

print("Loading embedding model and building the FAISS index (first run downloads the model)...")
vector_store = VectorStore(chunks)
print(f"Indexed {len(chunks)} chunks into FAISS.")


Loading embedding model and building the FAISS index (first run downloads the model)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 58 chunks into FAISS.


## Step 3: Metadata filtering + vector search (retrieval)

`vector_store.search()` first filters chunks by brand and model, then embeds the query and finds the most similar chunks within that subset.

That filter matters because otherwise a question about the Hyundai Creta could pull a similar chunk from another brochure.

In [4]:
results = vector_store.search("What is the mileage?", brand="Hyundai", model="Creta", top_k=5)
for r in results:
    print(f"[{r['chunk']['section']} | page {r['chunk']['page_number']}] score={r['score']:.3f}")
    print(r['chunk']['text'][:100], "...\n")


[safety | page 15] score=0.392
Key features
Engine & trim plan
1.5 l MPi petrol 
MT 
 
● 
● 
● 
● 
● 
● 
● 
● 
-
 
iVT 
 
- 
- 
● 
 ...

[engine and performance | page 17] score=0.379
Dimensions
1 790
Technical specifications
4 330
2 610
Dimensions 
 
 
Overall length (mm) 
 
4 330 
 ...

[infotainment and connectivity | page 6] score=0.371
Always forward thinking.
Stand above it all and yet stay completely connected with your world. When  ...

[interior and comfort | page 16] score=0.368
^ SE stands for summer edition
^^ Leatherette
* iVT/AT/DCT only
** Works with compatible smartphones ...

[safety | page 7] score=0.367
Forward Collision - Avoidance Assist
Junction Turning (FCA-JT)
Forward Collision - Avoidance Assist
 ...



## Step 4: Re-ranking + context window control

`src/reranker.py` uses the `cross-encoder/ms-marco-MiniLM-L-6-v2` model to re-score the shortlist from Step 3.

The cross-encoder reads the query and chunk together, so it is better at judging whether a chunk actually answers the question. We only run it on a small shortlist because it is slower than the first search step.

After re-ranking, only the best chunk is passed to the LLM.

In [5]:
from src.reranker import rerank

print("Loading cross-encoder re-ranking model...")
best_chunks = rerank("What is the mileage?", results, top_n=1)
for r in best_chunks:
    print(f"Best match: [{r['chunk']['section']}] rerank_score={r['rerank_score']:.3f}")


Loading cross-encoder re-ranking model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Best match: [engine and performance] rerank_score=-10.809


## Step 5: Answer generation + source attribution

`src/generator.py` builds a prompt with the best chunk as context and asks `google/flan-t5-small` to answer using only that text. This keeps the answer tied to the brochure instead of the model's training data.

The output includes the answer plus source details: brand, model, section, page number, and source file.

In [6]:
from src.generator import generate_answer

print("Loading generation model (flan-t5-small)...")
result = generate_answer("What is the mileage?", best_chunks)
print("Answer:", result["answer"])
print("Sources:", result["sources"])


Loading generation model (flan-t5-small)...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer: 205 / 65 R16 (D=405.6 mm) steel
Sources: [{'brand': 'Hyundai', 'model': 'Creta', 'section': 'engine and performance', 'page_number': 17, 'brochure_file': 'hyundai_creta.pdf', 'chunk_reference': 'hyundai_creta.pdf::engine and performance::p17', 'relevance_score': -10.809}]


### Calibrating the relevance threshold

`pipeline.ask()` compares the top re-ranker score with `min_rerank_score`. If the score is too low, it returns "I don't have information about that" instead of forcing an answer.

Run the cell below to compare an in-scope and out-of-scope question, then pick a `min_rerank_score` value between the two scores.

In [7]:
in_scope = rerank("What is the mileage?", vector_store.search("What is the mileage?", brand="Hyundai", model="Creta", top_k=5), top_n=1)
out_of_scope = rerank("What is the price?", vector_store.search("What is the price?", brand="Hyundai", model="Creta", top_k=5), top_n=1)

print("In-scope question rerank_score:", in_scope[0]["rerank_score"] if in_scope else None)
print("Out-of-scope question rerank_score:", out_of_scope[0]["rerank_score"] if out_of_scope else None)
print("Pick a min_rerank_score threshold between these two numbers in src/pipeline.py")


In-scope question rerank_score: -10.809427261352539
Out-of-scope question rerank_score: -10.861966133117676
Pick a min_rerank_score threshold between these two numbers in src/pipeline.py


## Step 6: The full pipeline (all steps wired together)

`src/pipeline.py` wraps Steps 1 to 5 in a `DriveWisePipeline` class with a simple `.ask(brand, model, query)` interface. It also applies the threshold check and logs every query.

This is the same class used by `app_web.py`, so the notebook and the web app use the same logic. Adding a new car just means dropping another `.txt` brochure into `data/brochures/`.

In [8]:
from src.pipeline import DriveWisePipeline

pipeline = DriveWisePipeline()
print("Available cars:", pipeline.list_available_cars())


[chunker] Loaded PDF: hyundai_creta.pdf  (16 chunks)
[chunker] Loaded PDF: maruti_baleno.pdf  (8 chunks)
[chunker] Loaded PDF: tata_nexon.pdf  (34 chunks)


Available cars: [('Hyundai', 'Creta'), ('Maruti Suzuki', 'Baleno'), ('Tata', 'Nexon')]


## Step 7: Logging and monitoring

Every `pipeline.ask()` call writes one JSON line to `logs/query_log.jsonl`. Each entry stores the query, the retrieved chunks, the response time, and whether the request succeeded.

JSONL is a good fit here because new entries can be appended one line at a time and each line can be read on its own.

The cell below runs three demo queries and then reads the log back to confirm the entries were written.

In [9]:
r1 = pipeline.ask("Hyundai", "Creta", "What is the mileage of the diesel variant?")
print(r1["answer"])
print(r1["sources"])
print()

r2 = pipeline.ask("Tata", "Nexon", "Does it have electronic stability control?")
print(r2["answer"])
print(r2["sources"])
print()

# An out-of-scope question should fail gracefully, not invent an answer
r3 = pipeline.ask("Maruti Suzuki", "Baleno", "What is the price of this car?")
print(r3["answer"])
print(r3["sources"])


205 / 65 R16 (D=405.6 mm) steel
[{'brand': 'Hyundai', 'model': 'Creta', 'section': 'engine and performance', 'page_number': 17, 'brochure_file': 'hyundai_creta.pdf', 'chunk_reference': 'hyundai_creta.pdf::engine and performance::p17', 'relevance_score': -8.024}]

I couldn't find anything about that in the brochure for this car. Try rephrasing your question or check you selected the right model.
[]



PG_6
[{'brand': 'Maruti Suzuki', 'model': 'Baleno', 'section': 'interior and comfort', 'page_number': 4, 'brochure_file': 'maruti_baleno.pdf', 'chunk_reference': 'maruti_baleno.pdf::interior and comfort::p4', 'relevance_score': -8.909}]


In [10]:
import json
with open("logs/query_log.jsonl") as f:
    for line in f:
        print(json.dumps(json.loads(line), indent=2))


{
  "timestamp": "2026-07-19T08:05:18.807991+00:00",
  "brand": "Hyundai",
  "model": "Creta",
  "query": "What is the mileage of the diesel variant?",
  "retrieval_results": [
    {
      "chunk_reference": "hyundai_creta.txt::mileage and fuel efficiency::p3",
      "section": "mileage and fuel efficiency",
      "page_number": 3,
      "relevance_score": 5.261
    }
  ],
  "num_chunks_found": 1,
  "response_time_seconds": 12.5818,
  "answer_generation_status": "success",
  "failed": false
}
{
  "timestamp": "2026-07-22T05:49:35.136999+00:00",
  "brand": "Hyundai",
  "model": "Creta",
  "query": "What is the mileage?",
  "retrieval_results": [],
  "num_chunks_found": 0,
  "response_time_seconds": 11.2841,
  "answer_generation_status": "failed_no_relevant_data",
  "failed": true
}
{
  "timestamp": "2026-07-22T05:52:09.327555+00:00",
  "brand": "Hyundai",
  "model": "Creta",
  "query": "What is the mileage?",
  "retrieval_results": [],
  "num_chunks_found": 0,
  "response_time_seconds":

## Step 8: Evaluation (answer correctness, faithfulness, context relevance)

`src/evaluator.py` runs the pipeline on a small test set of five hand-written questions where the expected keyword and correct section are already known. It checks three things:

- Answer correctness: does the generated answer include the expected keyword?
- Faithfulness: does the answer stay within what the retrieved chunk actually says?
- Context relevance: did retrieval return a chunk that matches the question?

These checks give a quick picture of where the pipeline works well and where it needs tuning.

In [11]:
from src.evaluator import run_evaluation

def ask_fn(brand, model, query):
    return pipeline.ask(brand, model, query)

run_evaluation(ask_fn)


Query                                         Correct?   Faithful   Ctx.Rel.  
--------------------------------------------------------------------------------


What is the mileage of the diesel variant?    False      1.0        0.0       
How many airbags does it have?                False      1.0        1.0       


What is the boot space?                       False      0.0        0.0       


Does it have a sunroof?                       False      0.0        1.0       
What is the ground clearance?                 False      1.0        1.0       
--------------------------------------------------------------------------------
Overall Answer Correctness: 0% (0/5)


## Step 9: Improvements from evaluation feedback

Three areas were flagged in the project evaluation:
1. **Input validation** – reject bad queries before they enter the pipeline
2. **Fuzzy car search** – find a car by typing its name, not just a number
3. **Pipeline error handling** – each stage is wrapped so failures surface as
   a descriptive `PipelineError`, not a raw Python traceback

All three are implemented in `src/validator.py` and wired into
`src/pipeline.py`, `app.py`, and `app_web.py`.

### 9a: Input validation

`src/validator.validate_query()` strips whitespace, then rejects:
- empty strings
- purely numeric or symbol-only input (e.g. `"???"`, `"123"`)
- strings with fewer than 3 word-characters

It returns the cleaned query on success, or raises `ValueError` with a
user-facing message on failure. Both `app.py` and `app_web.py` call this
before touching the pipeline.

In [12]:
from src.validator import validate_query

test_inputs = [
    "What is the mileage?",   # valid
    "",                        # empty
    "   ",                     # whitespace only
    "???",                     # symbols only
    "123",                     # digits only
    "ab",                      # too short (< 3 word-chars)
    "What is the boot space?", # valid
]

print(f"{'Input':<35} {'Result':<10} {'Value / Error'}")
print("-" * 75)
for q in test_inputs:
    try:
        clean = validate_query(q)
        print(f"{repr(q):<35} {'PASS':<10} {repr(clean)}")
    except ValueError as e:
        print(f"{repr(q):<35} {'FAIL':<10} {e}")


Input                               Result     Value / Error
---------------------------------------------------------------------------
'What is the mileage?'              PASS       'What is the mileage?'
''                                  FAIL       Query cannot be empty. Please type a question about the car.
'   '                               FAIL       Query cannot be empty. Please type a question about the car.
'???'                               FAIL       Query contains only symbols or numbers. Please ask a proper question, e.g. 'What is the mileage?'
'123'                               FAIL       Query contains only symbols or numbers. Please ask a proper question, e.g. 'What is the mileage?'
'ab'                                FAIL       Query is too short (need at least 3 letters). Please be more specific.
'What is the boot space?'           PASS       'What is the boot space?'


### 9b: Fuzzy car search

Previously, `app.py` only accepted a number to pick a car. Now it also
accepts a text search term.  `pipeline.fuzzy_search_cars()` uses Python's
built-in `difflib.SequenceMatcher` (no extra dependencies) to rank all
available cars by similarity to the typed query.

Substring and prefix matches are boosted, so short inputs like `"creta"`
or `"tata"` still produce confident results.

In [13]:
# pipeline was already initialised in Step 6
# pipeline = DriveWisePipeline()  # uncomment if running this cell standalone

search_terms = ["creta", "nexon", "tata", "maruti", "balen", "hyund", "xyz"]

for term in search_terms:
    matches = pipeline.fuzzy_search_cars(term, top_n=3)
    print(f"Query: {repr(term):<12} → top matches:")
    for m in matches:
        print(f"   {m['label']:<25} score={m['score']:.0%}")
    print()


Query: 'creta'      → top matches:
   Hyundai Creta             score=60%
   Tata Nexon                score=27%
   Maruti Suzuki Baleno      score=16%

Query: 'nexon'      → top matches:
   Tata Nexon                score=67%
   Hyundai Creta             score=22%
   Maruti Suzuki Baleno      score=16%

Query: 'tata'       → top matches:
   Tata Nexon                score=60%
   Hyundai Creta             score=24%
   Maruti Suzuki Baleno      score=17%

Query: 'maruti'     → top matches:
   Maruti Suzuki Baleno      score=60%
   Hyundai Creta             score=32%
   Tata Nexon                score=25%

Query: 'balen'      → top matches:
   Maruti Suzuki Baleno      score=60%
   Tata Nexon                score=40%
   Hyundai Creta             score=22%

Query: 'hyund'      → top matches:
   Hyundai Creta             score=60%
   Maruti Suzuki Baleno      score=16%
   Tata Nexon                score=13%

Query: 'xyz'        → top matches:
   Tata Nexon                score=15%
   Hyund

### 9c: Pipeline error handling

`src/pipeline.py` now wraps each stage in its own `try/except` block.
Any failure raises `PipelineError(stage, cause)` instead of letting a
raw exception bubble up to the user.

| Stage | Triggered by |
|---|---|
| `initialisation` | Missing brochures folder, bad PDF |
| `retrieval` | FAISS index corruption |
| `reranking` | Cross-encoder model error |
| `generation` | LLM failure |
| `logging` | Disk full / permissions error |

The cell below simulates a retrieval failure to show what the error
looks like and that it is caught cleanly.

In [14]:
from src.validator import PipelineError
from unittest.mock import patch

# Simulate a retrieval-stage failure by making vector_store.search() raise
def _broken_search(*args, **kwargs):
    raise RuntimeError("Simulated FAISS index corruption")

with patch.object(pipeline.store, "search", _broken_search):
    try:
        pipeline.ask("Hyundai", "Creta", "What is the mileage?")
    except PipelineError as pe:
        print(f"Caught PipelineError")
        print(f"  stage : {pe.stage}")
        print(f"  cause : {pe.cause}")
        print(f"  full  : {pe}")
        print()
        print("The app.py / app_web.py layer translates this into a friendly")
        print("one-line message without showing a raw Python traceback.")


Caught PipelineError
  stage : retrieval
  cause : Simulated FAISS index corruption
  full  : [retrieval] RuntimeError: Simulated FAISS index corruption

The app.py / app_web.py layer translates this into a friendly
one-line message without showing a raw Python traceback.


## Conclusion

This notebook shows the DriveWise pipeline end to end: loading brochures, chunking them, retrieving relevant context, generating answers with citations, logging queries, and running a small evaluation.

The main design choices were:
- Filter by metadata before similarity search so answers stay car-specific
- Chunk by section so topics stay intact
- Use a fast bi-encoder first and a cross-encoder second
- Apply a relevance threshold so out-of-scope questions fail gracefully

The same `src/` modules also power the Streamlit app in `app_web.py`, so both interfaces use the same logic.